<a href="https://colab.research.google.com/github/Tejas7575/ML-Projects/blob/main/Agentic_AI_Research_analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# !ollama --version # Ollama is not installed in the Colab environment.

In [5]:
!pip install crewai crewai-tools  -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.2/197.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.4/836.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8

In [6]:
import os
import time
from crewai import Agent , Task, Crew, Process, LLM
from crewai_tools import SerperDevTool

In [7]:
os.environ["SERPER_API_KEY"] = "d2849ea93d134c6651e1e0ee3b2cee1bec053425"

In [15]:
import google.generativeai as genai
from google.colab import userdata

# Configure Gemini API
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Use Gemini as the LLM for CrewAI
llm = LLM(
    model=genai.GenerativeModel('gemini-1.5-flash'), # Or 'gemini-1.5-pro' for more capabilities
    temperature=0.7
)

search_tool = SerperDevTool(n_results=3)

SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

In [16]:
# Cell 3 — agents
research_analyst = Agent(
    role='Research Analyst',
    goal='Find latest news and 2 stats about {topic}',
    backstory="Research analyst who cites sources and uses recent data.",
    tools=[search_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=4,
)

blog_writer = Agent(
    role='LinkedIn Writer',
    goal='Write a 400-word LinkedIn post from research',
    backstory="Viral LinkedIn writer, conversational tone, uses hooks and CTAs.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=4,
)

In [17]:
# Cell 4 — tasks
research_task = Task(
    description="""Research the latest developments, news, and stats about: {topic}
    Focus on: 1. Key trends 2. 3 Important news headlines 3. 2 Data points/Stats
    Use search to verify.""",
    expected_output="""A structured research summary with:
      - 3 Key Trends
      - 3 News Headlines with links
      - 2 Stats with sources""",
    agent=research_analyst
)

writing_task = Task(
    description="""Using the research summary, write a 400-word LinkedIn blog post about {topic}.
    Structure: Hook -> 3 Key Insights -> What it means for professionals -> CTA
    Tone: Professional but conversational. Add 3 relevant hashtags.""",
    expected_output="A complete 400-word blog post ready to publish",
    agent=blog_writer,
    context=[research_task]
)

In [14]:
# Cell 5 — crew & run
crew = Crew(
    agents=[research_analyst, blog_writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

topic = "AI in Hyderabad 2026"
print(f"\n### Starting Research and Writing Crew for: {topic} ###\n")
result = crew.kickoff(inputs={"topic": topic})
print("\n\n### FINAL BLOG POST ###\n")
print(result)


### Starting Research and Writing Crew for: AI in Hyderabad 2026 ###



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9d1ac4dc-806a-494f-8786-124999c1e086                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Research the latest developments, news, and stats about: AI in Hyderabad 2026                            │
│      Focus on: 1. Key trends 2. 3 Important news headlines 3. 2 Data points/Stats                               │
│      Use search to verify.                                                                                      │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the latest developments, news, and stats about: AI in Hyderabad 2026                            │
│      Focus on: 1. Key trends 2. 3 Important news headlines 3. 2 Data points/Stats                               │
│      Use search to verify.                                                                                      │
│  ID: ff350a30-51ba-4365-add0-89d3e30eeaac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [18]:
# Cell 5 — crew & run
crew = Crew(
    agents=[research_analyst, blog_writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

topic = "AI in Hyderabad 2026"
print(f"\n### Starting Research and Writing Crew for: {topic} ###\n")
result = crew.kickoff(inputs={"topic": topic})
print("\n\n### FINAL BLOG POST ###\n")
print(result)


### Starting Research and Writing Crew for: AI in Hyderabad 2026 ###



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e908d1ee-72d0-4980-b4ed-95c5ac2c9722                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the latest developments, news, and stats about: AI in Hyderabad 2026                            │
│      Focus on: 1. Key trends 2. 3 Important news headlines 3. 2 Data points/Stats                               │
│      Use search to verify.                                                                                      │
│  ID: 5ee6475f-191b-467b-a4b9-22d49a383d8b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Research the latest developments, news, and stats about: AI in Hyderabad 2026                            │
│      Focus on: 1. Key trends 2. 3 Important news headlines 3. 2 Data points/Stats                               │
│      Use search to verify.                                                                                      │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: e908d1ee-72d0-4980-b4ed-95c5ac2c9722                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯